# 04_Machine_Learning: Model Building and Evaluation

This notebook builds and evaluates machine learning models to detect suspicious transactions, comparing several algorithms and saving the best performing model.


## Setup and Load Data

Import the cleaned dataset and prepare for machine learning.

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE

DATA_PROCESSED_DIR = Path('data/processed')
df = pd.read_csv(DATA_PROCESSED_DIR / 'cleaned_transactions.csv')


## Prepare Features and Target

Select feature columns, encode categorical variables, and split the data into training and testing sets.

In [ ]:

# Define target and features
TARGET_COL = 'is_laundering'

# Drop rows with missing target values
if df[TARGET_COL].isnull().any():
    df = df.dropna(subset=[TARGET_COL])

# Identify numeric and categorical columns
numeric_cols = df.select_dtypes(include=['int64','float64']).columns.tolist()
categorical_cols = df.select_dtypes(include=['object','category']).columns.tolist()

if TARGET_COL in numeric_cols:
    numeric_cols.remove(TARGET_COL)
if TARGET_COL in categorical_cols:
    categorical_cols.remove(TARGET_COL)

X = df[numeric_cols + categorical_cols]
y = df[TARGET_COL]

# Preprocess: scale numeric features and one-hot encode categorical features
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ]
)

# Apply SMOTE for balancing classes
smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X, y)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_res, y_res, test_size=0.3, random_state=42, stratify=y_res)

# Fit preprocessor on training data and transform
X_train_pre = preprocessor.fit_transform(X_train)
X_test_pre = preprocessor.transform(X_test)



## Model Training and Evaluation

Train baseline and advanced models, evaluate their performance, and compare metrics.

In [ ]:

# Function to evaluate and print metrics
def evaluate_model(name, model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = None
    if hasattr(model, 'predict_proba'):
        y_prob = model.predict_proba(X_test)[:, 1]
    metrics = {
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1': f1_score(y_test, y_pred)
    }
    if y_prob is not None:
        metrics['ROC_AUC'] = roc_auc_score(y_test, y_prob)
    return metrics

# Initialize models
log_reg = LogisticRegression(max_iter=1000, class_weight='balanced')
rf = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42)

# Evaluate
results = []
for name, model in [('Logistic Regression', log_reg), ('Random Forest', rf)]:
    metrics = evaluate_model(name, model, X_train_pre, y_train, X_test_pre, y_test)
    results.append(metrics)

# Convert to DataFrame
results_df = pd.DataFrame(results)
print(results_df)


## Save Metrics and Model

Save evaluation metrics, model comparison, and the trained model for future use.

In [ ]:

from pathlib import Path
import joblib

MODELS_DIR = Path('models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Save metrics to markdown
metrics_md_path = MODELS_DIR / 'model_metrics.md'
with open(metrics_md_path, 'w') as f:
    f.write('# Model Evaluation Metrics

')
    f.write(results_df.to_markdown(index=False))

# Save comparison to CSV
results_df.to_csv(MODELS_DIR / 'model_comparison.csv', index=False)

# Save best model (e.g., Random Forest)
best_model = rf
joblib.dump(best_model, MODELS_DIR / 'trained_model.joblib')
print('Saved model, metrics, and comparison file.')


## Business Interpretation

Discuss the implications of model performance metrics. Recall and F1-score are particularly important because false negatives (missing suspicious transactions) can have serious consequences.

In [ ]:

print('Interpretation:
')
print('- High recall indicates the model is effective at catching suspicious transactions, reducing false negatives.')
print('- Precision indicates the proportion of flagged transactions that are actually suspicious, balancing operational workload.')
print('- F1-score balances precision and recall, providing a single metric to compare models.')
